# Exact range mass through turn → river

Given one turn and two weighted ranges, calculate every river's probability, updated compatible pair mass, conditional beliefs, weighted W/L/T and equity. Then verify exact mass conservation and the law of total equity.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from fractions import Fraction
import pandas as pd
from cards import hand_names, make_hand
from hand_space import PRIVATE_HANDS
from mass import (
    RangeMass, compatible_joint_mass, range_equity, range_is_invariant,
    transport_range, turn_mass, weighted_signature,
)

## 1. Unnormalised exact ranges

The first range gives twice as much mass to top pair as to its weak hand. The second gives three times as much mass to its pair as to its weak hand. Each range still occupies the fixed 1,326-coordinate ambient basis.

In [2]:
turn = make_hand(['2c', '7d', '9h', 'Js'])
first_strong = make_hand(['Jc', 'Ad'])
first_weak = make_hand(['4d', '5d'])
second_strong = make_hand(['9c', 'Kd'])
second_weak = make_hand(['8c', 'Qd'])
first_range = RangeMass.from_hands({first_strong: 2, first_weak: 1})
second_range = RangeMass.from_hands({second_strong: 3, second_weak: 1})

print('Turn:', ' '.join(hand_names(turn)))
print('Range vector lengths:', len(first_range.weights), len(second_range.weights))
print('Unnormalised totals:', first_range.total, second_range.total)
print('Nonzero coordinates:', len(first_range.support), len(second_range.support))

Turn: 2c 7d 9h Js
Range vector lengths: 1326 1326
Unnormalised totals: 3 4
Nonzero coordinates: 2 2


## 2. Compatible product joint mass

The sparse joint model contains only compatible ordered hand pairs. Its weights are the products of the two marginal range weights.

In [3]:
parent = compatible_joint_mass(turn, first_range, second_range)
joint_table = pd.DataFrame([
    {
        'first hand': ' '.join(hand_names(PRIVATE_HANDS[entry.first_index])),
        'second hand': ' '.join(hand_names(PRIVATE_HANDS[entry.second_index])),
        'weight': entry.weight,
        'conditional probability': entry.weight / parent.total,
    }
    for entry in parent.entries
])
print('Compatible joint mass:', parent.total)
joint_table

Compatible joint mass: 12


,first hand,second hand,weight,conditional probability
0,Jc Ad,8c Qd,2,1/6
1,Jc Ad,9c Kd,6,1/2
2,4d 5d,8c Qd,1,1/12
3,4d 5d,9c Kd,3,1/4


## 3. Transport mass through all rivers

Each compatible hand pair distributes its weight uniformly across its 44 legal rivers. Summing those contributions produces range-dependent river probabilities. All 48 public river labels remain present.

In [4]:
result = turn_mass(turn, first_range, second_range)
river_table = pd.DataFrame([
    {
        'river': river.river_name,
        'probability': river.probability,
        'distributed mass': river.joint_mass.total,
        'conditional pairs': len(river.joint_mass.entries),
        'weighted wins': river.signature.wins,
        'weighted losses': river.signature.losses,
        'weighted ties': river.signature.ties,
        'equity': river.signature.equity if river.probability else None,
    }
    for river in result.rivers
])
river_table

,river,probability,distributed mass,conditional pairs,weighted wins,weighted losses,weighted ties,equity
0,3c,1/44,3/11,4,2/11,1/11,0,2/3
1,4c,1/44,3/11,4,9/44,3/44,0,3/4
2,5c,1/44,3/11,4,9/44,3/44,0,3/4
3,6c,1/44,3/11,4,2/11,1/11,0,2/3
4,7c,1/44,3/11,4,2/11,1/11,0,2/3
5,8c,3/176,9/44,2,3/22,3/44,0,2/3
6,9c,1/176,3/44,2,1/22,1/44,0,2/3
7,Tc,1/44,3/11,4,3/22,3/22,0,1/2
8,Jc,1/132,1/11,2,0,1/11,0,0
9,Qc,1/44,3/11,4,3/22,3/22,0,1/2


## 4. Blockers make river probabilities range-dependent

A card appearing in a high-weight hand blocks more joint mass than a card appearing only in a low-weight hand. Consequently the river distribution need not be uniform.

In [5]:
display(river_table.sort_values('probability').head(8))
display(river_table.sort_values('probability', ascending=False).head(8))
print('Distinct river probabilities:', river_table['probability'].nunique())

,river,probability,distributed mass,conditional pairs,weighted wins,weighted losses,weighted ties,equity
22,Kd,1/176,3/44,2,1/22,1/44,0,2/3
6,9c,1/176,3/44,2,1/22,1/44,0,2/3
23,Ad,1/132,1/11,2,0,1/11,0,0
8,Jc,1/132,1/11,2,0,1/11,0,0
15,5d,1/66,2/11,2,2/11,0,0,1
14,4d,1/66,2/11,2,2/11,0,0,1
21,Qd,3/176,9/44,2,3/22,3/44,0,2/3
5,8c,3/176,9/44,2,3/22,3/44,0,2/3


,river,probability,distributed mass,conditional pairs,weighted wins,weighted losses,weighted ties,equity
0,3c,1/44,3/11,4,2/11,1/11,0,2/3
1,4c,1/44,3/11,4,9/44,3/44,0,3/4
26,4h,1/44,3/11,4,9/44,3/44,0,3/4
27,5h,1/44,3/11,4,9/44,3/44,0,3/4
28,6h,1/44,3/11,4,2/11,1/11,0,2/3
29,7h,1/44,3/11,4,2/11,1/11,0,2/3
30,8h,1/44,3/11,4,2/11,1/11,0,2/3
31,Th,1/44,3/11,4,3/22,3/22,0,1/2


Distinct river probabilities: 5


## 5. Exact conservation and total equity

The 48 child masses must sum to the parent mass. River probabilities must sum to one. Finally, averaging conditional river equity with those probabilities must reproduce the aggregate turn equity.

In [6]:
child_mass_sum = sum((river.joint_mass.total for river in result.rivers), Fraction())
probability_sum = sum((river.probability for river in result.rivers), Fraction())

print('Parent mass:', result.parent_mass.total)
print('Sum of child masses:', child_mass_sum)
print('Sum of river probabilities:', probability_sum)
print('Aggregate turn equity:', result.equity, f'= {float(result.equity):.2%}')
print('Probability-weighted river equity:', result.average_river_equity, f'= {float(result.average_river_equity):.2%}')
assert child_mass_sum == result.parent_mass.total
assert probability_sum == 1
assert result.average_river_equity == result.equity

Parent mass: 12
Sum of child masses: 12
Sum of river probabilities: 1
Aggregate turn equity: 157/264 = 59.47%
Probability-weighted river equity: 157/264 = 59.47%


## 6. Range mass can break suit symmetry

Chance children may be suit-isomorphic while their weighted masses are not. A weighted orbit may be multiplied by its chance multiplicity only when the relevant range or joint belief is invariant under that symmetry.

In [7]:
swap_clubs_diamonds = (1, 0, 2, 3)
moved_range = transport_range(first_range, swap_clubs_diamonds)

print('Original range is invariant:', range_is_invariant(first_range, swap_clubs_diamonds))
print('Total mass after transport:', moved_range.total)
print('Transport changes coordinates:', moved_range != first_range)

Original range is invariant: False
Total mass after transport: 3
Transport changes coordinates: True


## Summary

This is the bridge from chance structure to beliefs: exact marginal ranges produce compatible pair mass; chance distributes that mass; blockers determine child probabilities; every child receives updated conditional beliefs and a weighted payoff signature; and the entire calculation obeys exact mass and equity identities.